# 🧠 Notebook 16: State Space Models (SSMs)

Beyond attention — learning sequence models that scale linearly with sequence length, from classical control theory through S4 to Mamba.

**Prerequisites:** Notebooks 05 (Self-Attention), 08 (Training on Apple Silicon)

**What you'll learn:**
- 💡 Why attention's O(n²) cost becomes a bottleneck for long sequences
- 🎯 The continuous-time SSM formulation: dx/dt = Ax + Bu, y = Cx
- ⚡ How SSMs achieve O(n) complexity via linear recurrence
- ⚠️ The tradeoff: SSMs gain speed but lose the ability to "look back" at arbitrary positions

## The Problem: Attention is O(n²)

Self-attention is the engine of the transformer. For every token in a sequence of length $n$, it computes a similarity score against **every other token**. That means:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

The $QK^\top$ product creates an $n \times n$ attention matrix. Both the **compute** and **memory** scale quadratically:

| Resource | Scaling | At n=4,096 | At n=131,072 |
|----------|---------|------------|-------------|
| FLOPs (QK^T) | O(n² · d) | ~25 GFLOPs | ~25 TFLOPs |
| Memory (attn matrix) | O(n²) | 64 MB | 64 GB |

For a model with $d_{\text{model}} = 768$:
- At sequence length **4,096**: the attention matrix is 4,096 × 4,096 = **16.7M entries** → 64 MB in float32
- At sequence length **131,072** (a book): 131,072 × 131,072 = **17.2B entries** → 64 GB in float32

That 64 GB exceeds the unified memory of most Apple Silicon machines. Even with Flash Attention (Notebook 12), the **compute** is still O(n²) — we just avoid materializing the full matrix.

💡 **Key insight:** Attention's power comes from letting every token attend to every other token. But this all-pairs comparison is exactly what makes it expensive. Can we find a model that processes sequences **one step at a time** instead?

## The Alternative: Linear Recurrence — O(n)

RNNs had the right idea: process tokens sequentially, maintaining a **hidden state** that summarizes everything seen so far:

$$h_t = f(h_{t-1}, x_t)$$

This is O(n) in sequence length — each step does constant work. But classical RNNs (and even LSTMs) had two fatal problems:

1. **Vanishing/exploding gradients** — information decays exponentially over long distances
2. **Sequential bottleneck** — can't parallelize across the sequence during training

State Space Models solve both problems by using a **structured linear recurrence** with carefully designed matrices. The key insight: if $f$ is **linear**, the recurrence can be computed as a **convolution** during training (parallelizable!) and as a **recurrence** during inference (O(1) per step).

⚡ **The SSM promise:** O(n) compute, O(1) memory per step at inference, and parallelizable training via convolution.

| Property | Attention | RNN/LSTM | SSM |
|----------|-----------|----------|-----|
| Training complexity | O(n²) | O(n) sequential | O(n log n) parallel |
| Inference per step | O(n) (KV cache) | O(1) | O(1) |
| Memory at inference | O(n) (KV cache grows) | O(1) | O(1) |
| Long-range modeling | Excellent | Poor | Good (with selection) |
| Parallelizable training | ✅ | ❌ | ✅ |

---

## Math Derivation: The Continuous-Time State Space Model

Following the pedagogical pattern: **math first**, then code, then visualization.

SSMs come from **control theory** and **signal processing**, where they've been used for decades to model dynamical systems (think: circuits, mechanical systems, audio filters). The core idea: model a system with a hidden state that evolves continuously over time.

### The State Equation

A continuous-time SSM is defined by two equations:

$$\frac{dx}{dt} = Ax(t) + Bu(t) \qquad \text{(state equation)}$$

$$y(t) = Cx(t) \qquad \text{(output equation)}$$

where:
- $u(t) \in \mathbb{R}^{D}$ is the **input signal** at time $t$ (e.g., one feature of a token embedding)
- $x(t) \in \mathbb{R}^{N}$ is the **hidden state** — a compressed summary of the input history
- $y(t) \in \mathbb{R}^{D}$ is the **output signal**
- $A \in \mathbb{R}^{N \times N}$ is the **state transition matrix** — controls how the state evolves
- $B \in \mathbb{R}^{N \times D}$ is the **input projection matrix** — maps input into state space
- $C \in \mathbb{R}^{D \times N}$ is the **output projection matrix** — maps state back to output space

🎯 **Interview tip:** The letters A, B, C come directly from control theory notation. If an interviewer asks "why these letters?", it's because SSMs are literally borrowed from classical control.

### Shape Analysis

Let's be precise about dimensions. For a single input channel (the SSM operates independently per channel):

| Symbol | Shape | Role |
|--------|-------|------|
| $u(t)$ | scalar (1,) | Input at time $t$ for one channel |
| $x(t)$ | $(N,)$ | Hidden state vector |
| $y(t)$ | scalar (1,) | Output at time $t$ for one channel |
| $A$ | $(N, N)$ | State transition — how state dimensions interact |
| $B$ | $(N, 1)$ | Input-to-state projection |
| $C$ | $(1, N)$ | State-to-output projection |

In practice, we run **one SSM per channel** of the $D$-dimensional input. So for a batch of sequences:

| Tensor | Shape | Notes |
|--------|-------|-------|
| Input $u$ | $(B, L, D)$ | Batch × Sequence length × Channels |
| State $x$ | $(B, D, N)$ | One $N$-dim state per channel |
| Output $y$ | $(B, L, D)$ | Same shape as input |
| $A$ | $(D, N)$ | Diagonal structure (one per channel) |
| $B$ | $(D, N)$ | Per-channel input projection |
| $C$ | $(D, N)$ | Per-channel output projection |

💡 **Key insight:** The state dimension $N$ is typically small (16 or 64). The "width" of the model comes from running $D$ independent SSMs in parallel — one per channel. This is what keeps the model efficient.

### Why This Formulation Works

The continuous-time ODE $\frac{dx}{dt} = Ax + Bu$ has a closed-form solution:

$$x(t) = e^{At}x(0) + \int_0^t e^{A(t-\tau)} Bu(\tau)\, d\tau$$

This tells us:
1. The state **decays** according to $e^{At}$ — if $A$ has negative eigenvalues, old information fades (like a low-pass filter)
2. New input is **accumulated** through the integral — weighted by how recently it arrived
3. The matrix $A$ controls the **timescale** of memory — its eigenvalues determine how fast information decays

⚠️ **Pitfall:** If $A$ has positive eigenvalues, the state **explodes** exponentially. In practice, SSMs initialize $A$ with negative real parts (e.g., the HiPPO matrix) to ensure stability.

### Connection to Signal Processing

This is exactly a **linear time-invariant (LTI) system** — the same math used in:
- **Audio processing:** Infinite impulse response (IIR) filters
- **Control systems:** PID controllers, Kalman filters
- **Circuit analysis:** RLC circuits

The key difference in deep learning: we **learn** the matrices $A$, $B$, $C$ from data, rather than designing them by hand. And in Mamba (which we'll build later), we make $B$ and $C$ **input-dependent**, breaking the LTI assumption to gain selectivity.

### From Continuous to Discrete: The Bridge to Code

We can't run a continuous ODE on a computer — we need to **discretize** it. Given a step size $\Delta$ (which can be learned!), we convert:

$$\frac{dx}{dt} = Ax + Bu \quad \xrightarrow{\text{discretize}} \quad x_t = \bar{A}\, x_{t-1} + \bar{B}\, u_t$$

Using **Zero-Order Hold (ZOH)** discretization:

$$\bar{A} = e^{\Delta A} \qquad \bar{B} = (\Delta A)^{-1}(e^{\Delta A} - I) \cdot \Delta B$$

For computational efficiency (and because $A$ is often diagonal), we use the first-order approximation:

$$\bar{A} \approx e^{\Delta A} \qquad \bar{B} \approx \Delta B$$

This gives us a **discrete recurrence** we can implement:

$$x_t = \bar{A} \cdot x_{t-1} + \bar{B} \cdot u_t$$
$$y_t = C \cdot x_t$$

🎯 **Interview tip:** The step size $\Delta$ controls the "resolution" of the SSM. A small $\Delta$ means fine-grained temporal resolution (good for audio); a large $\Delta$ means coarse resolution (good for summarizing long documents). In Mamba, $\Delta$ is **input-dependent** — the model learns when to pay attention and when to skip.

*We'll implement this discretization in the next section (Task 2.2).*

---

## Setup

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

from utils.ssm import plot_attention_vs_ssm_scaling, print_scaling_table

mx.random.seed(42)
print(f"MLX version: {mx.__version__}")
print("✅ Ready for State Space Models!")

---

## Visualizing the Quadratic Bottleneck

Let's make the O(n²) vs O(n) difference concrete with actual numbers. We'll compare attention and SSM scaling across sequence lengths from 128 to 131,072.

In [ ]:
# === Concrete numbers: Attention vs SSM at various sequence lengths ===
print_scaling_table()

The next cell continues the implementation:

**=== Visualization: O(n²) Attention vs O(n) SSM ===**

In [ ]:
# === Visualization: O(n²) Attention vs O(n) SSM ===
fig = plot_attention_vs_ssm_scaling()
plt.show()

### 🎯 Key Takeaways from the Scaling Comparison

The plots tell a clear story:

1. **At short sequences (≤512):** Attention is fine. The quadratic cost is manageable, and attention's ability to compare all pairs of tokens is a net win.

2. **At medium sequences (1K–8K):** Attention starts to hurt. The memory for the attention matrix grows fast, and tricks like Flash Attention help with memory but not compute.

3. **At long sequences (16K+):** Attention becomes impractical without heavy engineering. The attention matrix alone would consume gigabytes of memory. This is where SSMs shine — their cost grows linearly, and their memory is **constant** in sequence length.

⚡ **Performance insight:** At sequence length 131,072 (a full book), attention needs ~25 TFLOPs and 64 GB just for the attention matrix. An SSM needs ~25 GFLOPs and 48 KB of state. That's a **1,000× gap** in both compute and memory.

*Next up: we'll implement the SSM discretization and scan algorithm in MLX (Task 2.2).*

---

## Implementing the Simple SSM in MLX

Now let's turn the math into code. We'll implement a `SimpleSSM` module that:

1. **Discretizes** continuous parameters $(A, B)$ into discrete $\bar{A}, \bar{B}$ using Zero-Order Hold
2. **Scans** through the sequence with the recurrence $h_t = \bar{A} h_{t-1} + \bar{B} x_t$
3. **Projects** the hidden state to output: $y_t = C \cdot h_t$

💡 **Key design choice:** The SSM operates *per-channel* — one independent SSM with state dimension $N$ runs for each of the $D_{\text{inner}}$ channels. This keeps the model efficient while giving each channel its own dynamics.

⚠️ **Stability:** We initialize $A$ with negative values $(-1, -2, \ldots, -N)$ inspired by the HiPPO matrix. After discretization, $\bar{A} = e^{\Delta A}$ has magnitude $\leq 1$, preventing the hidden state from exploding.

In [ ]:
# === Import SimpleSSM from our utils module ===
from utils.ssm import SSMConfig, SimpleSSM

# Create a config — d_inner defaults to d_model * expand_factor
config = SSMConfig(d_model=64, d_state=16)
print(f"SSMConfig:")
print(f"  d_model      = {config.d_model}")
print(f"  d_state (N)  = {config.d_state}")
print(f"  d_inner (D)  = {config.d_inner}")
print(f"  expand_factor = {config.expand_factor}")
print(f"  dt_rank      = {config.dt_rank}")

The next cell continues the implementation:

**=== Create the SimpleSSM and inspect its parameters ===**

In [ ]:
# === Create the SimpleSSM and inspect its parameters ===
ssm = SimpleSSM(config)

print("SimpleSSM parameters:")
print(f"  A_log (state matrix)  : {ssm.A_log.shape}  — diagonal, negative values")
print(f"  B (input projection)  : {ssm.B.shape}  — learnable")
print(f"  C (output projection) : {ssm.C.shape}  — learnable")
print(f"  log_delta (step size) : {ssm.log_delta.shape}  — learnable (softplus for positivity)")
print(f"\n💡 Total state per channel: {config.d_state} floats")
print(f"   Total state for all channels: {config.d_inner} × {config.d_state} = {config.d_inner * config.d_state} floats")

### Zero-Order Hold Discretization

The discretization step converts continuous-time parameters to discrete-time:

$$\bar{A} = e^{\Delta \cdot A} \qquad \bar{B} = \Delta \cdot B$$

Since $A$ is diagonal with negative values and $\Delta > 0$, every element of $\bar{A}$ satisfies $|\bar{A}_{ij}| \leq 1$. This guarantees the recurrence is **stable** — the hidden state can't explode.

🎯 **Interview tip:** The step size $\Delta$ controls temporal resolution. Small $\Delta$ → fine-grained (audio). Large $\Delta$ → coarse (long documents). In Mamba, $\Delta$ becomes input-dependent.

In [ ]:
# === Demonstrate discretization: A -> A_bar, B -> B_bar ===
delta = nn.softplus(ssm.log_delta)  # ensure positivity
mx.eval(delta)

A_bar, B_bar = SimpleSSM.discretize(ssm.A_log, ssm.B, delta)
mx.eval(A_bar, B_bar)

print("Discretization results:")
print(f"  delta (step size) range: [{delta.min().item():.4f}, {delta.max().item():.4f}]")
print(f"  A (continuous)    range: [{ssm.A_log.min().item():.1f}, {ssm.A_log.max().item():.1f}]")
print(f"  A_bar (discrete)  range: [{A_bar.min().item():.6f}, {A_bar.max().item():.6f}]")
print(f"  |A_bar| max magnitude:   {mx.max(mx.abs(A_bar)).item():.6f}  ⚡ (must be ≤ 1.0 for stability)")
print(f"  B_bar (discrete)  range: [{B_bar.min().item():.6f}, {B_bar.max().item():.6f}]")
print(f"\n✅ A_bar magnitude ≤ 1.0: {bool(mx.all(mx.abs(A_bar) <= 1.0).item())}")

### Sequential Scan: The SSM Recurrence

With discrete parameters in hand, we run the recurrence:

$$h_t = \bar{A} \cdot h_{t-1} + \bar{B} \cdot x_t \qquad y_t = C \cdot h_t$$

This processes the sequence **one step at a time**, maintaining a hidden state $h \in \mathbb{R}^{D_{\text{inner}} \times N}$. The memory is constant in sequence length — only the current state matters.

⚡ **Performance note:** The sequential scan is O(n) in sequence length but inherently serial. During training, this can be parallelized via the *parallel scan* algorithm (Task 2.4). For inference, the sequential scan is actually ideal — O(1) per token.

In [ ]:
# === Forward pass: verify input/output shapes ===
batch, seq_len = 2, 32
x = mx.random.normal((batch, seq_len, config.d_inner))
print(f"Input shape:  {x.shape}  — [batch={batch}, seq={seq_len}, d_inner={config.d_inner}]")

y = ssm(x)
mx.eval(y)
print(f"Output shape: {y.shape}  — [batch={batch}, seq={seq_len}, d_inner={config.d_inner}]")
print(f"\n✅ Shape preserved: {x.shape == y.shape}")
print(f"✅ No NaN: {not bool(mx.any(mx.isnan(y)).item())}")
print(f"✅ No Inf: {not bool(mx.any(mx.isinf(y)).item())}")
print(f"\nOutput stats:")
print(f"  mean = {y.mean().item():.6f}")
print(f"  std  = {mx.sqrt(mx.mean(y * y)).item():.6f}")
print(f"  min  = {y.min().item():.6f}")
print(f"  max  = {y.max().item():.6f}")

The next cell continues the implementation:

**=== Verify causality: output at t depends only on inputs 0..t ===**

In [ ]:
# === Verify causality: output at t depends only on inputs 0..t ===
# Changing input at position t+1 should NOT affect output at position t
mx.random.seed(123)
x_base = mx.random.normal((1, 8, config.d_inner))

# Run on original input
y_base = ssm(x_base)
mx.eval(y_base)

# Modify input at position 5 (should not affect outputs 0-4)
x_modified = mx.array(x_base)  # copy
x_mod_np = x_modified.tolist()
for i in range(config.d_inner):
    x_mod_np[0][5][i] = 999.0
x_modified = mx.array(x_mod_np)

y_modified = ssm(x_modified)
mx.eval(y_modified)

# Check that outputs at positions 0-4 are identical
max_diff_before = mx.max(mx.abs(y_base[:, :5, :] - y_modified[:, :5, :])).item()
max_diff_at = mx.max(mx.abs(y_base[:, 5, :] - y_modified[:, 5, :])).item()

print("Causality check:")
print(f"  Max diff at positions 0-4 (before change): {max_diff_before:.2e}  (should be ~0)")
print(f"  Max diff at position 5 (changed input):    {max_diff_at:.2e}  (should be > 0)")
print(f"\n✅ Causal: {max_diff_before < 1e-5 and max_diff_at > 1e-5}")

### 🎯 Key Takeaways: SimpleSSM

1. **Discretization** converts continuous $(A, B)$ to discrete $(\bar{A}, \bar{B})$ via $\bar{A} = e^{\Delta A}$, $\bar{B} = \Delta B$
2. **Sequential scan** runs the recurrence $h_t = \bar{A} h_{t-1} + \bar{B} x_t$ in O(n) time
3. **Per-channel operation** — each of the $D_{\text{inner}}$ channels has its own independent SSM
4. **Stability** — negative $A$ values guarantee $|\bar{A}| \leq 1$, preventing state explosion
5. **Constant memory** — hidden state is $(B, D_{\text{inner}}, N)$ regardless of sequence length

⚠️ **Limitation:** This SimpleSSM uses *fixed* parameters — $A$, $B$, $C$ don't depend on the input. This means it's a Linear Time-Invariant (LTI) system and can't selectively focus on different parts of the input. Mamba (Task 2.4) fixes this by making $\Delta$, $B$, $C$ input-dependent.

*Next: Property tests for discretization correctness and causality (Task 2.3), then the Selective SSM (Task 2.4).*

---

## From Fixed to Selective: The Mamba Innovation

The SimpleSSM above is a **Linear Time-Invariant (LTI)** system — its parameters $A$, $B$, $C$, $\Delta$ are the same regardless of what input it sees. This is like having a fixed audio filter: it processes every sound the same way.

But language isn't like audio. Some tokens matter a lot ("not" in "this is not good"), while others are filler ("the", "a"). We need the model to **selectively** decide what to remember and what to forget.

💡 **The Mamba insight:** Make $\Delta$, $B$, and $C$ **input-dependent**. Project them from the input at each timestep:

$$\Delta_t, B_t, C_t = \text{project}(x_t)$$

This breaks the LTI assumption — the system now adapts its behaviour based on what it sees:

| Parameter | Role when input-dependent |
|-----------|--------------------------|
| $\Delta_t$ (step size) | **When to pay attention.** Large $\Delta$ → integrate this input strongly. Small $\Delta$ → ignore it, keep previous state. |
| $B_t$ (input matrix) | **What to remember.** Controls how the current input maps into the state. |
| $C_t$ (output matrix) | **What to output.** Controls which parts of the state are read out. |

🎯 **Interview tip:** The key difference between S4 and Mamba is selectivity. S4 uses fixed parameters (LTI), which means it can be computed as a convolution during training. Mamba uses input-dependent parameters, which requires a scan — but gains the ability to do content-based reasoning.

⚠️ **Tradeoff:** By making parameters input-dependent, we lose the convolution view (no more FFT trick). Instead, we need an efficient **parallel scan** algorithm for training.

### SelectiveSSM Architecture

The `SelectiveSSM` projects input-dependent parameters through a combined linear projection:

```
x ──→ x_proj ──→ [dt_input | B | C]
                      │
                      ▼
                   dt_proj ──→ softplus ──→ Δ (per-channel step size)
```

**Discretization** (same ZOH as before, but now per-timestep):

$$\bar{A}_t = e^{\Delta_t \cdot A}, \qquad \bar{B}_t = \Delta_t \cdot B_t$$

**Sequential scan** (used at inference — O(1) per step):

$$h_t = \bar{A}_t \cdot h_{t-1} + \bar{B}_t \cdot x_t, \qquad y_t = C_t \cdot h_t$$

**Parallel scan** (used at training — O(n log n) work, O(log n) depth):

The recurrence $h_t = \bar{A}_t h_{t-1} + \bar{B}_t x_t$ is an **associative operation** on tuples $(a, b)$:

$$(a_2, b_2) \circ (a_1, b_1) = (a_2 \cdot a_1, \; a_2 \cdot b_1 + b_2)$$

This means we can compute all hidden states in parallel using a **prefix sum** (Blelloch scan), trading O(n) serial work for O(n log n) parallel work with only O(log n) depth.

⚡ **Performance:** Sequential scan is ideal for inference (process one token at a time). Parallel scan is ideal for training (process the whole sequence at once on GPU).

In [ ]:
# === Import and create SelectiveSSM ===
from utils.ssm import SelectiveSSM

selective_ssm = SelectiveSSM(config)

print("SelectiveSSM parameters:")
print(f"  A_log (state matrix)  : {selective_ssm.A_log.shape}  — diagonal, negative values")
print(f"  x_proj (combined proj): Linear({config.d_inner} → {config.dt_rank + 2*config.d_state})")
print(f"    → dt_input: {config.dt_rank} dims")
print(f"    → B:        {config.d_state} dims (input-dependent!)")
print(f"    → C:        {config.d_state} dims (input-dependent!)")
print(f"  dt_proj (Δ projection): Linear({config.dt_rank} → {config.d_inner})")
print(f"\n💡 Key difference from SimpleSSM: Δ, B, C are projected from input at each timestep")

The next cell continues the implementation:

**=== Forward pass with sequential scan (inference mode) ===**

In [ ]:
# === Forward pass with sequential scan (inference mode) ===
batch, seq_len = 2, 32
x = mx.random.normal((batch, seq_len, config.d_inner))

y_seq = selective_ssm(x, use_parallel=False)
mx.eval(y_seq)

print(f"Input shape:  {x.shape}  — [batch={batch}, seq={seq_len}, d_inner={config.d_inner}]")
print(f"Output shape: {y_seq.shape}  — [batch={batch}, seq={seq_len}, d_inner={config.d_inner}]")
print(f"\n✅ Shape preserved: {x.shape == y_seq.shape}")
print(f"✅ No NaN: {not bool(mx.any(mx.isnan(y_seq)).item())}")
print(f"✅ No Inf: {not bool(mx.any(mx.isinf(y_seq)).item())}")

### Verifying Sequential vs Parallel Scan Equivalence

The sequential and parallel scans compute the same recurrence — just with different parallelism strategies. They must produce identical outputs (within floating-point tolerance).

⚡ **Why both?** Sequential scan is O(n) work, O(n) depth — great for inference where you process one token at a time. Parallel scan is O(n log n) work, O(log n) depth — great for training where you want to use all GPU cores on the full sequence.

In [ ]:
# === Compare sequential vs parallel scan outputs ===
y_par = selective_ssm(x, use_parallel=True)
mx.eval(y_par)

max_diff = mx.max(mx.abs(y_seq - y_par)).item()
mean_diff = mx.mean(mx.abs(y_seq - y_par)).item()

print("Sequential vs Parallel scan comparison:")
print(f"  Max absolute difference:  {max_diff:.2e}")
print(f"  Mean absolute difference: {mean_diff:.2e}")
print(f"  Tolerance threshold:      1.00e-05")
print(f"\n✅ Equivalent within 1e-5: {max_diff < 1e-5}")

# Also test with a non-power-of-2 sequence length
x_odd = mx.random.normal((1, 13, config.d_inner))
y_seq_odd = selective_ssm(x_odd, use_parallel=False)
y_par_odd = selective_ssm(x_odd, use_parallel=True)
mx.eval(y_seq_odd, y_par_odd)
diff_odd = mx.max(mx.abs(y_seq_odd - y_par_odd)).item()
print(f"\n🎯 Non-power-of-2 (seq_len=13): max diff = {diff_odd:.2e}, within 1e-5: {diff_odd < 1e-5}")

### 🎯 Key Takeaways: SelectiveSSM

1. **Input-dependent parameters** — $\Delta$, $B$, $C$ are projected from the input, breaking the LTI assumption
2. **Selective attention** — the model learns WHEN to update state (large $\Delta$) and WHEN to ignore input (small $\Delta$)
3. **Two scan modes** — sequential (O(n) serial, for inference) and parallel (O(n log n), for training)
4. **Associative scan** — the recurrence can be expressed as a prefix sum with the operator $(a_2, b_2) \circ (a_1, b_1) = (a_2 a_1, a_2 b_1 + b_2)$
5. **Numerical equivalence** — both scans produce identical results within floating-point tolerance (~1e-7)

⚠️ **What's still missing:** The full Mamba block wraps SelectiveSSM with input projection, 1D convolution, SiLU gating, and output projection. That's Task 2.6 — coming up next.

*Next: Property tests for scan equivalence and state boundedness (Task 2.5), then the full MambaBlock (Task 2.6).*

---

## The Full Mamba Block

The `SelectiveSSM` is the engine, but a real Mamba layer wraps it with several additional components. The full `MambaBlock` follows this pipeline:

```
x ─→ RMSNorm ─→ in_proj ─→ split ─→ [x_ssm | z (gate)]
                                        │          │
                                     conv1d      SiLU
                                        │          │
                                      SiLU         │
                                        │          │
                                       SSM         │
                                        │          │
                                        └─── × ────┘
                                             │
                                          out_proj
                                             │
                                        + residual
                                             │
                                           output
```

💡 **Why each component matters:**

| Component | Purpose | Analogy |
|-----------|---------|--------|
| **RMSNorm** | Stabilize inputs before processing | Same as pre-norm in transformers |
| **in_proj** | Expand `d_model` → `2 × d_inner` (SSM + gate branches) | Like QKV projection in attention |
| **conv1d** | Local context mixing (kernel size 4) | Like a tiny attention window of size 4 |
| **SelectiveSSM** | Global context via linear recurrence | Like attention but O(n) |
| **SiLU gating** | Control information flow between branches | Like GLU in SwiGLU FFN |
| **out_proj** | Compress `d_inner` → `d_model` | Like output projection in attention |
| **Residual** | Gradient highway, prevents degradation | Same as in transformers |

🎯 **Interview tip:** The conv1d is a key Mamba design choice. It provides *local* context (the last 4 tokens) cheaply, while the SSM provides *global* context (the entire history). Together they cover both short-range and long-range dependencies.

⚠️ **Causal conv1d:** The convolution pads on the **left** only, so position $t$ sees only positions $t-3, t-2, t-1, t$ (for kernel size 4). No future leakage.

### 🎯 Key Takeaways: MambaBlock

1. **Full pipeline:** RMSNorm → in_proj → split → conv1d → SiLU → SSM → gate → out_proj → residual
2. **Two branches:** The input projection creates an SSM branch (processed by conv1d + SSM) and a gate branch (just SiLU). The gate controls what information flows through.
3. **Local + global context:** Conv1d provides a small local window (4 tokens). The SSM provides global context across the entire sequence.
4. **Shape preservation:** Input `[batch, seq, d_model]` → Output `[batch, seq, d_model]` — drop-in replacement for a transformer block
5. **Constant memory:** The SSM state is `O(batch × d_inner × d_state)` — independent of sequence length

⚡ **Performance comparison at seq_len = 131,072:**
- Transformer attention: ~64 GB for the attention matrix alone
- MambaBlock SSM state: ~16 KB (with d_inner=128, d_state=16)
- That's a **4,000,000× difference** in memory!

*Next: Attention vs SSM benchmarks (Task 2.7), then the history of sequence models (Task 2.8).*

In [ ]:
# === Import and create MambaBlock ===
from utils.ssm import MambaBlock

# Use the same config — MambaBlock wraps SelectiveSSM
mamba = MambaBlock(config)

print("MambaBlock components:")
print(f"  norm:     RMSNorm({config.d_model})")
print(f"  in_proj:  Linear({config.d_model} → {2 * config.d_inner})  [SSM branch + gate branch]")
print(f"  conv1d:   Causal depthwise, kernel_size={config.d_conv}, channels={config.d_inner}")
print(f"  ssm:      SelectiveSSM(d_inner={config.d_inner}, d_state={config.d_state})")
print(f"  out_proj: Linear({config.d_inner} → {config.d_model})")
print(f"\n💡 Input/output shape: [batch, seq, {config.d_model}]")
print(f"   Internal expanded dim: {config.d_inner} (expand_factor={config.expand_factor})")

The next cell continues the implementation:

**=== Verify memory is independent of sequence length ===**

In [ ]:
# === Verify memory is independent of sequence length ===
print("Memory analysis: SSM state size vs sequence length")
print(f"{'Seq Length':>10} | {'Output Shape':>20} | {'SSM State Size':>15} | {'Status'}")
print("-" * 70)

for seq_len in [32, 128, 512, 1024, 2048]:
    x_test = mx.random.normal((2, seq_len, config.d_model))
    y_test = mamba(x_test)
    mx.eval(y_test)
    
    # SSM state is always batch × d_inner × d_state (in float32)
    state_bytes = 2 * config.d_inner * config.d_state * 4
    
    assert y_test.shape == x_test.shape, f"Shape mismatch at seq_len={seq_len}"
    assert not bool(mx.any(mx.isnan(y_test)).item()), f"NaN at seq_len={seq_len}"
    
    print(f"{seq_len:>10,} | {str(y_test.shape):>20} | {state_bytes:>11,} B | ✅")

state_kb = 2 * config.d_inner * config.d_state * 4 / 1024
print(f"\n💡 SSM state is always {state_kb:.1f} KB = batch({2}) × d_inner({config.d_inner}) × d_state({config.d_state}) × 4 bytes")
print(f"   This is CONSTANT regardless of sequence length!")
print(f"\n   Compare: attention KV cache at seq_len=2048 would be:")
attn_kv_bytes = 2 * 2048 * config.d_model * 4 * 2  # 2 for K+V, 2 for batch
print(f"   {attn_kv_bytes:,} bytes = {attn_kv_bytes/1024:.1f} KB  (and grows linearly!)")

The next cell continues the implementation:

**=== Forward pass: verify input shape == output shape ===**

In [ ]:
# === Forward pass: verify input shape == output shape ===
batch, seq_len = 2, 32
x = mx.random.normal((batch, seq_len, config.d_model))  # Note: d_model, not d_inner!
print(f"Input shape:  {x.shape}  — [batch={batch}, seq={seq_len}, d_model={config.d_model}]")

y = mamba(x)
mx.eval(y)
print(f"Output shape: {y.shape}  — [batch={batch}, seq={seq_len}, d_model={config.d_model}]")

print(f"\n✅ Shape preserved (input == output): {x.shape == y.shape}")
print(f"✅ No NaN: {not bool(mx.any(mx.isnan(y)).item())}")
print(f"✅ No Inf: {not bool(mx.any(mx.isinf(y)).item())}")
print(f"\nOutput stats:")
print(f"  mean = {y.mean().item():.6f}")
print(f"  std  = {mx.sqrt(mx.mean((y - y.mean()) ** 2)).item():.6f}")
print(f"  min  = {y.min().item():.6f}")
print(f"  max  = {y.max().item():.6f}")

### Memory: O(batch × d_inner × d_state), Independent of Sequence Length

The key advantage of the Mamba architecture: the SSM hidden state is always `(batch, d_inner, d_state)` regardless of how long the sequence is. Compare this to attention, where the KV cache grows linearly with sequence length.

⚡ **This is why Mamba can handle 1M+ token sequences** — the state size is fixed at a few kilobytes, while attention's KV cache would need gigabytes.

---

## ⚡ Attention vs SSM: Head-to-Head Benchmarks

We've seen the *theoretical* scaling differences — O(n²) for attention vs O(n) for SSMs. Now let's measure *actual* wall-clock time and compare memory usage on real MLX computations.

We'll benchmark:
1. **Standard scaled dot-product attention** — computes the full n×n attention matrix
2. **MambaBlock** — our full Mamba implementation with conv1d + selective SSM + gating

💡 **What to watch for:** At small sequence lengths, attention may be competitive (or even faster) because it's a simple matrix multiply that hardware loves. The SSM advantage emerges as sequences get longer and the O(n²) cost starts to dominate.

⚠️ **Note:** These benchmarks use small d_model=64 for quick execution. In production models (d_model=4096+), the gaps would be much larger.

In [ ]:
# === Benchmark: Attention vs MambaBlock at various sequence lengths ===
from utils.ssm_benchmarks import benchmark_attention_vs_ssm, plot_benchmark_results, plot_quality_comparison

results = benchmark_attention_vs_ssm(
    d_model=64,
    seq_lengths=[64, 128, 256, 512, 1024],
    batch_size=2,
    n_warmup=2,
    n_runs=5,
)

# Print timing table
print(f"{'Seq Len':>8} | {'Attention (ms)':>14} | {'MambaBlock (ms)':>15} | {'Speedup':>8}")
print("-" * 55)
for sl, at, st in zip(results['seq_lengths'], results['attn_times_ms'], results['ssm_times_ms']):
    ratio = at / st if st > 0 else float('inf')
    marker = '✅' if ratio > 1 else '⚠️'
    print(f"{sl:>8} | {at:>14.2f} | {st:>15.2f} | {ratio:>6.2f}× {marker}")

The next cell continues the implementation:

**=== Visualization: Timing, Memory, and Speedup ===**

In [ ]:
# === Visualization: Timing, Memory, and Speedup ===
fig = plot_benchmark_results(results)
plt.show()

### 💾 Memory Scaling: The Real Story

The timing comparison tells part of the story, but **memory** is where SSMs truly shine:

| Metric | Attention | SSM (Mamba) |
|--------|-----------|-------------|
| **Attention/State memory** | O(n²) — grows quadratically | O(d_inner × d_state) — **constant** |
| **At seq_len=1,024** | 1,024 × 1,024 × 4 = **4 MB** | 128 × 16 × 4 = **8 KB** |
| **At seq_len=131,072** | 131K × 131K × 4 = **64 GB** | 128 × 16 × 4 = **8 KB** |
| **KV cache (inference)** | Grows linearly with context | Fixed state size |

⚡ **This is the killer advantage:** At 131K tokens, attention needs 64 GB just for the attention matrix, while the SSM state stays at 8 KB. That's an **8,000,000× difference**.

🎯 **Interview tip:** When asked "why not just use SSMs everywhere?", the answer is quality. Attention excels at tasks requiring precise retrieval or copying from context. SSMs compress everything into a fixed-size state, which means they can lose fine-grained details. This is why hybrid architectures (Jamba, Griffin) combine both.

In [ ]:
# === Quality Comparison: Where each architecture excels ===
fig = plot_quality_comparison()
plt.show()

### 🎯 Key Takeaways: Attention vs SSM Benchmarks

1. **Compute:** Attention is O(n²), SSM is O(n). At longer sequences, the SSM advantage grows dramatically.
2. **Memory:** Attention materializes an n×n matrix (or KV cache that grows linearly). SSM state is **constant** — a few KB regardless of sequence length.
3. **Quality tradeoffs:**
   - Attention wins at **retrieval** and **in-context learning** — it can look back at any position
   - SSM wins at **long-range** tasks — it can process 100K+ tokens without running out of memory
   - For **short sequences** (≤512), both are competitive
4. **The hybrid solution:** Modern architectures like Jamba (AI21, 2024) and Griffin (Google, 2024) interleave attention and SSM layers to get the best of both worlds.

⚠️ **Caveat:** These benchmarks use small models (d_model=64). In production-scale models, the O(n²) vs O(n) gap is even more pronounced because the constant factors in attention (QKV projections, softmax) scale with d_model.

*Next: The history of sequence models, from RNNs to Mamba-2 (Task 2.8).*

## 🧪 Try It Yourself

Experiment with State Space Models:

1. **Discretization**: Try different delta values (0.01, 0.1, 1.0). How does the discretized A_bar change? What happens when delta is very large?

2. **Causality check**: Modify a token in the middle of a sequence. Verify that only later outputs change (not earlier ones).

3. **Memory comparison**: For seq_len=1024, compare the memory needed for attention (O(n^2)) vs SSM (O(n * d_state)). At what sequence length does SSM win?

---
## ➡️ What's Next?

You've completed this notebook! In **Notebook 17: Alignment**, we'll explore RLHF, DPO, and GRPO for making models helpful.

💡 **Before moving on**, make sure you can answer these questions:
- What was the main concept in this notebook?
- Why does it matter for building LLMs?
- Could you explain it to a friend in one sentence?

---

## 📜 History & Alternatives

The quest for efficient sequence modeling is one of the longest-running threads in deep learning. From the earliest recurrent networks to today's hybrid architectures, each generation solved one problem while revealing the next. Here's how we got from RNNs to Mamba — and where the field is heading.

### Chronological Timeline

| Year | Model / Paper | Team | Key Contribution |
|------|--------------|------|-----------------|
| 1997 | **LSTM** | Hochreiter & Schmidhuber | Gated recurrence to fight vanishing gradients. Dominated sequence modeling for two decades. |
| 2017 | **Transformer** | Vaswani et al. (Google) | Self-attention replaces recurrence entirely. O(n²) but massively parallelizable. |
| 2020 | **Linear Attention** | Katharopoulos et al. (EPFL/Idiap) | Remove softmax from attention to get O(n) complexity via kernel trick: $\phi(Q)\phi(K)^T V$. Quality gap vs full attention remained significant. |
| 2021 | **S4** | Gu et al. (Stanford) | Structured State Spaces for Sequence Modeling. HiPPO initialization for long-range memory. First SSM to match transformers on Long Range Arena. Computed via convolution (FFT) during training. |
| 2022 | **H3** | Fu et al. (Stanford) | Hungry Hungry Hippos — combined SSM layers with a single attention layer. Showed SSMs alone struggle with recall tasks, but a hybrid closes the gap. |
| 2023 | **Hyena** | Poli et al. (Stanford) | Replaced attention with long convolutions parameterized by implicit neural representations. Subquadratic in sequence length, no attention needed. |
| 2023 | **Mamba** | Gu & Dao (Carnegie Mellon / Princeton) | Selective State Spaces — made $\Delta$, $B$, $C$ input-dependent, breaking the LTI constraint. Hardware-aware parallel scan on GPU. Matched transformer quality at 1.4B scale with 5× inference throughput. |
| 2024 | **RWKV-6** | Peng et al. (RWKV Foundation) | Linear attention RNN with data-dependent decay. Trained up to 14B parameters with competitive quality to transformers. Fully open-source with active community. |
| 2024 | **Griffin** | De et al. (Google DeepMind) | Gated linear recurrence (Real-Gated Linear Recurrence Unit) hybridized with local attention. Matched Llama-2 quality with better inference efficiency. |
| 2024 | **Mamba-2** | Dao & Gu | Structured State Space Duality (SSD) — proved SSMs and attention are mathematically dual. 2–8× faster than Mamba-1 via matrix-transfer algorithm. Unified framework connecting SSMs, linear attention, and structured matrices. |
| 2024 | **Jamba** | AI21 Labs | First production hybrid Mamba + attention architecture. Interleaved Mamba and attention layers with MoE. 256K context window, deployed commercially. Proved hybrid approach works at scale. |

### Key Trends

Several patterns emerge from this timeline:

**1. The recurrence ↔ attention pendulum**

The field keeps swinging between recurrent models (O(n), sequential) and attention models (O(n²), parallel). Each generation tries to get the best of both:
- RNNs → Transformers (traded recurrence for parallelism)
- Transformers → SSMs (traded parallelism for linear scaling)
- SSMs → Hybrids (combined both for the best tradeoff)

**2. The selectivity breakthrough**

The jump from S4 to Mamba was the single biggest advance. S4 used fixed (LTI) parameters — the same filter for every input. Mamba made parameters input-dependent, giving the model the ability to selectively remember or forget. This is analogous to how attention lets each token choose what to attend to.

**3. Hardware-aware algorithm design**

Mamba's parallel scan was co-designed with GPU memory hierarchy (similar to how Flash Attention was designed around SRAM/HBM). This trend of algorithm-hardware co-design is now standard — you can't separate the algorithm from the hardware it runs on.

**4. The hybrid consensus**

By 2024, the field converged on a practical answer: **use both**. Jamba, Griffin, and Mamba-2 all combine SSM layers with attention layers. The SSM layers handle long-range dependencies efficiently, while attention layers handle precise retrieval and in-context learning. This mirrors how the human brain uses both fast pattern matching (SSM-like) and deliberate recall (attention-like).

⚡ **Performance insight:** Hybrid architectures like Jamba achieve the best quality-efficiency tradeoff: SSM layers for the bulk of processing (cheap, O(n)), with a few attention layers sprinkled in for tasks that need exact recall (expensive but necessary).

### Alternatives to SSMs for Subquadratic Sequence Modeling

SSMs aren't the only approach to escaping O(n²) attention. Here's the landscape:

| Approach | Examples | Complexity | Key Idea | Limitation |
|----------|----------|-----------|----------|------------|
| **Sparse Attention** | Longformer, BigBird | O(n√n) | Attend to local windows + global tokens | Still quadratic in window size; pattern is fixed |
| **Linear Attention** | Katharopoulos et al., Performer | O(n) | Kernel approximation of softmax | Quality gap vs full attention; no causal masking trick |
| **State Space Models** | S4, Mamba, Mamba-2 | O(n) or O(n log n) | Linear recurrence with learned dynamics | Weaker at exact retrieval and in-context learning |
| **Long Convolutions** | Hyena, CKConv | O(n log n) | FFT-based global convolution | Less flexible than attention for variable-length reasoning |
| **Linear RNNs** | RWKV, RetNet, GLA | O(n) | Gated linear recurrence with decay | Training parallelism requires careful formulation |
| **Hybrid** | Jamba, Griffin, StripedHyena | O(n) + O(n²) sparse | Mix SSM/linear layers with a few attention layers | More complex architecture; harder to optimize |

💡 **Key insight:** No single approach dominates across all tasks. Attention excels at retrieval and in-context learning. SSMs excel at long-range dependencies and inference efficiency. The practical winner in 2024–2025 is the hybrid approach — and that's likely to remain true as models scale further.

### 🎯 Interview Tip: What to Memorize

If you're asked about SSMs in an interview, here are the key points to nail:

1. **The core equation:** $h_t = \bar{A} h_{t-1} + \bar{B} x_t$, $y_t = C h_t$ — a linear recurrence that's O(n) in sequence length
2. **Discretization:** $\bar{A} = e^{\Delta A}$, $\bar{B} = \Delta B$ — converts continuous-time ODE to discrete recurrence via Zero-Order Hold
3. **S4 vs Mamba:** S4 uses fixed parameters (LTI, can use FFT convolution). Mamba uses input-dependent $\Delta$, $B$, $C$ (selective, requires scan). Selectivity is what closed the quality gap with transformers.
4. **Parallel scan:** The recurrence is associative: $(a_2, b_2) \circ (a_1, b_1) = (a_2 a_1, a_2 b_1 + b_2)$. This enables O(log n) depth parallel computation during training.
5. **Mamba-2 duality:** SSMs and attention are mathematically dual — structured state space duality (SSD) connects them through structured matrices.
6. **Hybrid architectures:** Jamba and Griffin interleave SSM + attention layers. SSM handles long-range cheaply; attention handles retrieval precisely.
7. **Inference advantage:** SSM inference is O(1) per token with constant memory (just the hidden state). Attention inference is O(n) per token with growing KV cache.

⚠️ **Common interview mistake:** Don't say "SSMs replace attention." The correct framing is that SSMs are a *complementary* approach — modern architectures use both. The question isn't "which is better?" but "when do you use each?"